In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
import glob
from dataclasses import dataclass
import h5py as hf

In [ ]:
@dataclass
class Measurement:
    width: int = None
    depth: int = None
    entropy: float = None
    trace: float = None
    train_loss: np.ndarray = None
    test_loss: np.ndarray = None
    train_accuracy: np.ndarray = None,
    test_accuracy: np.ndarray = None,
    w_std: float = None
    b_std: float = None

In [ ]:
file_list = glob.glob("dense-*/*.h5")

In [ ]:
results = {}
for item in file_list:
    parts = item.split("/")[-1].split("_")
    arch = item.split("/")[0].split("-")
    name = f"{parts[0]}_{parts[1]}_{arch[0]}_{arch[1]}"
    
    if name not in list(results):
        results[name] = Measurement()
        results[name].w_std = parts[0]
        results[name].b_std = parts[1]
        results[name].width = arch[-1]
        results[name].depth = arch[1]
        
    if parts[2] == "train":
        with hf.File(item, "r") as db:
            loss = db["loss"][:]
            accuracy = db["accuracy"][:]
            
        results[name].train_loss = loss
        results[name].train_accuracy = accuracy
        results[name].width = arch[-1]
        results[name].depth = arch[1]
        
    elif parts[2] == "test":
        with hf.File(item, "r") as db:
            loss = db["loss"][:]
            accuracy = db["accuracy"][:]
            
        results[name].test_loss = loss
        results[name].test_accuracy = accuracy
        results[name].width = arch[-1]
        results[name].depth = arch[1]

    elif parts[2] == "cv":
        with hf.File(item, "r") as db:
            entropy = db["entropy"][:]
            trace = db["trace"][:]
            
        results[name].entropy = entropy
        results[name].trace = trace
        results[name].width = arch[-1]
        results[name].depth = arch[1]

In [ ]:
im = plt.scatter(
    x=[item.entropy for item in results.values() if min(item.test_loss) < 0.1],
    y=[float(item.depth) for item in results.values() if min(item.test_loss) < 0.1],
    c=[min(item.test_loss) for item in results.values() if min(item.test_loss) < 0.1],
    cmap="Spectral",
    norm=mpl.colors.LogNorm()
)
plt.colorbar(im)
plt.show()